In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    os.system('git clone https://github.com/vladlead5/tets.git /content/project')
    sys.path.insert(0, '/content/project')
    os.chdir('/content/project/notebooks')
    os.system('pip install tokenizers transformers sentencepiece -q')
else:
    sys.path.insert(0, os.path.abspath('..'))

print('Ready. CWD:', os.getcwd())


In [ ]:
import os
for d in ['../outputs/checkpoints', '../outputs/plots',
          '../outputs/metrics', '../outputs/generations']:
    os.makedirs(d, exist_ok=True)


# Лабораторная работа: Генеративные модели
## Часть 2: Word-level токенизация

В этом ноутбуке мы исследуем **пословную токенизацию** (word-level).

### Отличия от char-level:
- Каждый токен — слово или знак препинания
- Словарь намного больше (~5000–10000 токенов)
- Короткие последовательности (100 токенов ≈ ~600 символов)
- Редкие слова заменяются на `<UNK>`
- Эмбеддинги учатся сразу на уровне слов → лучше семантика

In [ ]:
import sys

import os
import json
import math
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from collections import Counter

from src.data.dataset import download_data, load_text, clean_text, train_val_split, TokenTextDataset
from src.tokenizers.word_tokenizer import WordTokenizer
from src.models.rnn_model import SimpleRNN
from src.models.lstm_model import LSTMModel
from src.models.bilstm_model import BiLSTMModel
from src.models.transformer_model import GPTModel
from src.training.trainer import train_model
from src.generation.generate import generate_rnn, generate_transformer
from src.evaluation.metrics import compute_perplexity
from src.utils.utils import set_seed, get_device, count_parameters

set_seed(42)
device = get_device()
print(f'Устройство: {device}')

## 1. Загрузка данных

In [ ]:
download_data('../data/shakespeare.txt')
raw_text = load_text('../data/shakespeare.txt')
text = clean_text(raw_text)[:500_000]

print(f'Символов: {len(text):,}')
print(f'Слов: {len(text.split()):,}')

## 2. Word-level токенизация

In [ ]:
tokenizer = WordTokenizer(max_vocab_size=8000, min_freq=2)
tokenizer.build_vocab(text)
print(f'vocab_size: {tokenizer.vocab_size}')

os.makedirs('../outputs/metrics', exist_ok=True)
tokenizer.save('../outputs/metrics/word_tokenizer.json')

In [ ]:
import re
all_words = re.findall(r'\b\w+\b|[^\w\s]', text.lower())
word_counts = Counter(all_words)

freq_data = sorted(word_counts.values(), reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].loglog(range(1, len(freq_data)+1), freq_data, 'b-', alpha=0.7)
axes[0].set_xlabel('Ранг слова (log)')
axes[0].set_ylabel('Частота (log)')
axes[0].set_title('Закон Zipf: распределение частот слов')
axes[0].grid(True, alpha=0.3)

top_words = word_counts.most_common(20)
words_disp = [w for w, _ in top_words]
freqs = [f for _, f in top_words]
axes[1].barh(range(len(words_disp)), freqs, color='steelblue')
axes[1].set_yticks(range(len(words_disp)))
axes[1].set_yticklabels(words_disp)
axes[1].set_xlabel('Частота')
axes[1].set_title('Топ-20 слов')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
os.makedirs('../outputs/plots', exist_ok=True)
plt.savefig('../outputs/plots/word_distribution.png', dpi=150)
plt.show()

total_tokens = sum(word_counts.values())
in_vocab = sum(f for w, f in word_counts.items() if w in tokenizer.word2idx)
print(f'\nОхват словаря: {in_vocab/total_tokens:.1%} токенов')
print(f'UNK-токены: {(total_tokens-in_vocab)/total_tokens:.1%} токенов')

In [ ]:
sample = 'To be, or not to be, that is the question.'
encoded = tokenizer.encode(sample)
decoded = tokenizer.decode(encoded)

print(f'Оригинал: {sample}')
print(f'Encoded ({len(encoded)} токенов): {encoded}')
print(f'Decoded:  {decoded}')

words_in_sample = re.findall(r'\b\w+\b|[^\w\s]', sample.lower())
print('\nМаппинг токенов:')
for word, idx in zip(words_in_sample, encoded):
    token_str = tokenizer.idx2word.get(idx, '<UNK>')
    print(f'  "{word}" → {idx} ({token_str})')

In [ ]:
train_text, val_text = train_val_split(text, val_fraction=0.1)

SEQ_LEN = 50
BATCH_SIZE = 64

train_ids = tokenizer.encode(train_text)
val_ids   = tokenizer.encode(val_text)

train_dataset = TokenTextDataset(train_ids, SEQ_LEN)
val_dataset   = TokenTextDataset(val_ids, SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f'Train: {len(train_ids):,} токенов → {len(train_dataset):,} примеров')
print(f'Val:   {len(val_ids):,} токенов → {len(val_dataset):,} примеров')

## 3. Обучение моделей

In [ ]:
VOCAB_SIZE = tokenizer.vocab_size
NUM_EPOCHS = 5

models_config = {
    'SimpleRNN': {
        'class': SimpleRNN,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn', 'lr': 1e-3,
    },
    'LSTM-1layer': {
        'class': LSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn', 'lr': 1e-3,
    },
    'LSTM-2layer': {
        'class': LSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn', 'lr': 1e-3,
    },
    'BiLSTM': {
        'class': BiLSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn', 'lr': 1e-3,
    },
    'GPT': {
        'class': GPTModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=256, num_heads=4, num_layers=4, max_seq_len=SEQ_LEN, dropout=0.1),
        'type': 'transformer', 'lr': 3e-4,
    },
}

print(f"{'Модель':20s} | {'Параметры':>12s}")
print('-' * 36)
for name, cfg in models_config.items():
    m = cfg['class'](**cfg['kwargs'])
    print(f'{name:20s} | {count_parameters(m):>12,}')

In [ ]:
all_histories = {}

for model_name, cfg in models_config.items():
    set_seed(42)
    model = cfg['class'](**cfg['kwargs'])
    
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        model_name=f'word_{model_name}',
        model_type=cfg['type'],
        num_epochs=NUM_EPOCHS,
        lr=cfg['lr'],
        checkpoint_dir='../outputs/checkpoints',
        device=device,
    )
    all_histories[model_name] = history

print('Обучение завершено!')

## 4. Результаты и сравнение

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

for (name, hist), color in zip(all_histories.items(), colors):
    epochs = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(epochs, hist['train_loss'], '--', color=color, alpha=0.7)
    axes[0].plot(epochs, hist['val_loss'], '-', color=color, label=name, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Кривые обучения (Word-level)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for (name, hist), color in zip(all_histories.items(), colors):
    epochs = range(1, len(hist['val_ppl']) + 1)
    axes[1].plot(epochs, hist['val_ppl'], '-o', color=color, label=name, markersize=4)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].set_title('Validation Perplexity (Word-level)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/plots/word_training_curves.png', dpi=150)
plt.show()

In [ ]:
rows = []
for name, cfg in models_config.items():
    m = cfg['class'](**cfg['kwargs'])
    hist = all_histories[name]
    best_val_loss = min(hist['val_loss'])
    rows.append({
        'Модель': name,
        'Параметры': f"{count_parameters(m):,}",
        'Best Val Loss': f"{best_val_loss:.4f}",
        'Best Val PPL': f"{compute_perplexity(best_val_loss):.1f}",
        'Сек/эпоха': f"{sum(hist['epoch_times'])/len(hist['epoch_times']):.1f}s",
    })

df = pd.DataFrame(rows)
print('Сравнение моделей (word-level токенизация):')
print(df.to_string(index=False))
df.to_csv('../outputs/metrics/word_comparison.csv', index=False)

## 5. Генерация текста

In [ ]:
def load_best(model_name, cfg, device):
    model = cfg['class'](**cfg['kwargs'])
    path = f'../outputs/checkpoints/word_{model_name}_best.pt'
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
    return model.to(device)

prompt = 'to be or not'
print('Генерация текста (word-level):\n')

all_generations = {}

for model_name, cfg in models_config.items():
    model = load_best(model_name, cfg, device)
    
    if cfg['type'] == 'transformer':
        gen = generate_transformer(model, tokenizer, prompt, max_new_tokens=50,
                                   strategy='temperature', temperature=0.8, device=device)
    else:
        gen = generate_rnn(model, tokenizer, prompt, max_new_tokens=50,
                           strategy='temperature', temperature=0.8, device=device)
    
    all_generations[model_name] = gen
    print(f'[{model_name}]')
    print(gen)
    print()

In [ ]:
if 'GPT' in models_config:
    model = load_best('GPT', models_config['GPT'], device)
    prompt = 'the king said'
    
    print(f'Промпт: "{prompt}"\n')
    for strategy in ['greedy', 'temperature', 'top_k']:
        gen = generate_transformer(model, tokenizer, prompt, max_new_tokens=40,
                                   strategy=strategy, device=device)
        print(f'[{strategy:12s}]: {gen}')

In [ ]:
with open('../outputs/generations/word_generations.txt', 'w', encoding='utf-8') as f:
    f.write('ГЕНЕРАЦИИ (word-level токенизация)\n' + '='*60 + '\n\n')
    for name, text in all_generations.items():
        f.write(f'[{name}]:\n{text}\n\n')

print('Готово!')

## 6. Выводы

### Word-level vs Char-level:

| Аспект | Char-level | Word-level |
|--------|------------|------------|
| Словарь | ~65 | ~5000–10000 |
| Длина последовательности | Длинная | Короткая |
| Семантика | Слабая | Сильная |
| OOV-проблема | Нет | Есть (UNK) |
| Грамматика | Приходится учить | Уже в токенах |

### Наблюдения:
- Word-level модели быстрее сходятся (меньше шагов на текст)
- Генерация более «связная» на уровне слов
- Проблема UNK: модель иногда теряет редкие слова
- PPL у word-level выше (больший словарь → сложнее предсказать)

Это **не значит** что word-level хуже — просто PPL несопоставима между разными словарями.